# lipsync-poc — 음성 기반 자동 페이셜 립싱크 데이터 생성기

성우 대사 음성(.wav)을 넣으면 **ARKit 52 블렌드쉐입 키프레임(30fps JSON/CSV)**을 만드는 PoC입니다.

이 노트북은 위에서부터 차례로 실행하면 됩니다 (메뉴: 런타임 → 모두 실행). GPU는 필요 없습니다.

1. 저장소 내려받기
2. 설치
3. 샘플 음성(TTS 10개)으로 립싱크 생성
4. 결과 보기 (그래프, 2D 입모양 영상, JSON)
5. 재현성 확인 (두 번 실행해 결과가 같은지)
6. 채점 지표 자체 검증
7. (선택) 직접 녹음한 목소리로 측정 결과 재현

처음 실행할 때 Whisper medium 모델(약 1.5GB)을 내려받습니다. 전체 실행에 5~10분쯤 걸립니다.

## 1. 저장소 내려받기

In [ ]:
import os

REPO = "https://github.com/benjy89a/Lip-sync.git"
if not os.path.exists("/content/Lip-sync"):
    !git clone --depth 1 {REPO} /content/Lip-sync
%cd /content/Lip-sync
!git log -1 --format="커밋 %h  %s"

## 2. 설치

- ffmpeg: 오디오 변환과 미리보기 영상 합성
- fonts-nanum: 그래프와 영상의 한글 표시
- 파이썬 패키지는 `requirements.txt`에 버전이 고정돼 있습니다.

아래 실행은 모두 `!python -m ...` (별도 프로세스)로 하므로, 설치 후 런타임을 다시 시작하지 않아도 됩니다.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg fonts-nanum > /dev/null
!pip -q install -r requirements.txt
!python -c "import nltk; [nltk.download(p, quiet=True) for p in ['averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'cmudict']]"
!python -c "import faster_whisper, ctranslate2, numpy; print('faster-whisper', faster_whisper.__version__, '| ctranslate2', ctranslate2.__version__, '| numpy', numpy.__version__)"
!ffmpeg -version | head -1

## 3. 립싱크 생성 (샘플 음성 10개)

`samples/audio/tts/`의 음성은 Windows 내장 TTS로 만든 샘플입니다. 대본은 `samples/script.csv`에 있습니다.

파일마다 다음이 출력됩니다: 인식된 문장, 단어 수, 키 수, 프레임 수, 생성 시간. `⚠ 검수 필요`는 Whisper가 확신하지 못한 단어가 있다는 표시입니다.

In [ ]:
!python -m src.cli samples/audio/tts

## 4. 결과 보기

`outputs/tts/<id>/`에 파일이 만들어집니다.

| 파일 | 내용 |
|---|---|
| `<id>.json` | 단어, 음소 구간, 입모양 키, ARKit 52채널 커브 |
| `<id>.csv` | 프레임별 52채널 값 |
| `<id>_curves.png` | 음량, 단어 구간, 채널별 커브 |
| `<id>_preview.mp4` | 2D 입모양 + 음성 |

In [ ]:
from IPython.display import Image, Video, display

SID = "ko_01"  # 다른 대사: ko_02 ~ ko_05, en_01 ~ en_03, fail_01, fail_02
display(Image(f"outputs/tts/{SID}/{SID}_curves.png"))

In [ ]:
display(Video(f"outputs/tts/{SID}/{SID}_preview.mp4", embed=True, width=720))

In [ ]:
import json

d = json.load(open(f"outputs/tts/{SID}/{SID}.json", encoding="utf-8"))
m = d["meta"]
print("인식된 문장:", m["transcript"])
print("언어:", m["language"], "| 길이:", m["duration"], "초 | 프레임:", m["frame_count"], "| fps:", m["fps"])
print("검수 필요:", m["confidence"]["needs_review"], "| 가장 낮은 단어 확률:", m["confidence"]["min_word_prob"])
print()
print("단어 시각:")
for w in d["words"]:
    print(f"  {w['start']:5.2f} ~ {w['end']:5.2f}  {w['text']}")
print()
print("입모양 키 (처음 12개): 시각, 그룹, 음소")
for k in d["keys"][:12]:
    print(f"  {k['time']:5.2f}  {k['group']:5s} {k['symbol']}")
print()
active = [name for name, v in d["curves"].items() if max(v) > 0]
print(f"값이 있는 채널 {len(active)}개 / 52개:", ", ".join(active))

전체 요약 (파일별 인식 결과와 단계별 소요 시간):

In [ ]:
import csv

rows = list(csv.DictReader(open("outputs/tts/summary.csv", encoding="utf-8")))
print(f"{'id':8s} {'언어':3s} {'길이':>5s} {'생성(초)':>7s} {'검수':4s} 인식된 문장")
for r in rows:
    gen = float(r["t_audio"]) + float(r["t_align"]) + float(r["t_viseme"])
    print(f"{r['id']:8s} {r['language']:3s} {float(r['duration']):5.2f} {gen:7.2f} {'⚠' if r['needs_review'] == 'True' else '-':4s} {r['transcript']}")

## 5. 재현성 확인

같은 입력으로 한 번 더 실행해서 JSON 결과가 완전히 같은지 해시로 비교합니다.

참고: **같은 환경에서** 다시 실행하면 결과가 같습니다. CPU 종류가 다른 환경(예: 로컬 PC와 Colab) 사이에서는
Whisper의 계산 결과가 미세하게 달라 수치가 조금 다를 수 있습니다.

In [ ]:
!python -m src.cli samples/audio/tts --out outputs/tts_again --no-preview > /dev/null 2>&1

import glob
import hashlib

def digest(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

same = 0
files = sorted(glob.glob("outputs/tts/*/*.json"))
for a in files:
    b = a.replace("outputs/tts/", "outputs/tts_again/")
    ok = digest(a) == digest(b)
    same += ok
    print("같음 " if ok else "다름 ", os.path.basename(a))
print(f"\n{same}/{len(files)}개 파일의 결과가 완전히 같습니다.")

## 6. 채점 지표 자체 검증

채점에 쓰는 지표가 계산을 제대로 하는지, **답을 알고 있는 합성 라벨**로 확인합니다.
예를 들어 예측을 일부러 3프레임 늦춘 라벨을 넣으면 평균 오차가 정확히 3.0이 나와야 합니다.

여기서 나오는 수치는 립싱크 품질이 아니라 지표의 계산이 맞는지만 보여 줍니다.

In [ ]:
!python -m src.selfcheck

## 7. (선택) 직접 녹음한 목소리로 측정 결과 재현

`docs/evaluation.md`의 결과 표는 직접 녹음한 대사 12개(`samples/audio/`)와 수작업 라벨(`samples/labels/`)로 낸 값입니다.
이 구간을 실행하면 그 표를 다시 계산합니다. 2~3분 걸립니다.

- `audio` : 기본 설정
- `audio_raw` : 단어 경계 보정을 끈 것
- `audio_rms` : 음량만으로 턱을 움직이는 기준선 (AI 없음)
- `audio_script` : 대본을 Whisper에 힌트로 준 것

In [ ]:
!python -m src.cli samples/audio --no-preview
!python -m src.cli samples/audio --no-refine-boundaries --out outputs/audio_raw --no-preview > /dev/null 2>&1
!python -m src.cli samples/audio --use-script --out outputs/audio_script --no-preview > /dev/null 2>&1
!python -m src.baseline samples/audio > /dev/null 2>&1
!python -m src.evaluate --human samples/labels/ko_01.txt samples/labels/repeat/ko_01.txt
!python -m src.evaluate outputs/audio outputs/audio_raw outputs/audio_rms outputs/audio_script > /dev/null

In [ ]:
rows = list(csv.DictReader(open("outputs/eval/audio_vs_audio_raw_vs_audio_rms_vs_audio_script_metrics.csv", encoding="utf-8-sig")))
print("라벨이 있는 대사 (ko_01, ko_05). 오차 단위는 프레임(30fps).\n")
print(f"{'실행':13s} {'id':6s} {'재현율':>6s} {'평균오차':>7s} {'2f이내':>6s} {'닫힘적중':>7s} {'잘못닫힘':>7s} {'수작업(초)':>9s} {'자동(초)':>8s}")
for r in rows:
    if r["id"] in ("ko_01", "ko_05"):
        print(f"{r['run']:13s} {r['id']:6s} {r['onset_recall']:>7s} {r['onset_mae_f']:>8s} {r['onset_within2_rate']:>7s} "
              f"{r['mbp_hit_rate']:>8s} {r['mbp_false_closed']:>8s} {r['manual_sec']:>10s} {r['auto_sec']:>9s}")

## 더 보기

- 문제 정의: `docs/problem_definition.md`
- 모델 선정 근거: `docs/model_selection.md`
- 검증 결과와 결론: `docs/evaluation.md`
- 회고: `docs/retrospective.md`

자기 음성으로 해 보려면 왼쪽 파일 창에서 wav/m4a 파일을 올리고 아래처럼 실행합니다.

```
!python -m src.cli /content/내파일.wav --lang ko
```